In [19]:
import pandas as pd

In [20]:
visits = pd.read_csv('visits.csv')
registrations = pd.read_csv('registrations.csv')
ads = pd.read_csv('ads.csv')

In [21]:
visits.head()

,uuid,platform,user_agent,date
0,1de9ea66-70d3-4a1f-8735-df5ef7697fb9,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_2...,2023-03-01T13:29:22
1,f149f542-e935-4870-9734-6b4501eaf614,web,Mozilla/5.0 (X11; CrOS x86_64 8172.45.0) Apple...,2023-03-01T16:44:28
2,f149f542-e935-4870-9734-6b4501eaf614,web,Mozilla/5.0 (X11; CrOS x86_64 8172.45.0) Apple...,2023-03-06T06:12:36
3,08f0ebd4-950c-4dd9-8e97-b5bdf073eed1,web,Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109...,2023-03-01T20:16:37
4,08f0ebd4-950c-4dd9-8e97-b5bdf073eed1,web,Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109...,2023-03-05T17:42:47


In [22]:
registrations.head()

,date,user_id,email,platform,registration_type
0,2023-03-01T00:25:39,8838849,joseph95@example.org,web,google
1,2023-03-01T14:53:01,8741065,janetsuarez@example.net,web,yandex
2,2023-03-01T14:27:36,1866654,robert67@example.org,web,google
3,2023-03-01T02:42:34,1577584,elam@example.net,web,apple
4,2023-03-01T10:27:14,4765395,stephanie68@example.net,web,yandex


In [23]:
ads.head()

,date,utm_source,utm_medium,utm_campaign,cost
0,2023-03-01T10:54:41,google,cpc,advanced_algorithms_series,212
1,2023-03-02T10:32:35,google,cpc,advanced_algorithms_series,252
2,2023-03-03T19:21:40,google,cpc,advanced_algorithms_series,202
3,2023-03-04T17:52:04,google,cpc,advanced_algorithms_series,223
4,2023-03-05T05:35:13,google,cpc,advanced_algorithms_series,265


In [24]:
visits = visits[~visits['user_agent'].str.contains('bot', case=False, na=False)].copy()

visits['date'] = pd.to_datetime(visits['date'])
registrations['date'] = pd.to_datetime(registrations['date'])
ads['date'] = pd.to_datetime(ads['date'])

visits['date_group'] = visits['date'].dt.date
registrations['date_group'] = registrations['date'].dt.date
ads['date_group'] = ads['date'].dt.date

visits = visits.sort_values('date')
visits = visits.drop_duplicates(subset='uuid', keep='last')


visits_grouped = (
    visits
    .groupby('date_group')
    .size()
    .reset_index(name='visits')
)

registrations_grouped = (
    registrations
    .groupby('date_group')
    .size()
    .reset_index(name='registrations')
)


df = pd.merge(
    visits_grouped,
    registrations_grouped,
    on='date_group',
    how='left'
)

df['registrations'] = df['registrations'].fillna(0)



ads_grouped = (
    ads
    .groupby(['date_group', 'utm_campaign'], as_index=False)
    .agg({'cost': 'sum'})
)


result = pd.merge(
    df,
    ads_grouped,
    on='date_group',
    how='left'
)

result['cost'] = result['cost'].fillna(0)
result['utm_campaign'] = result['utm_campaign'].fillna('none')


result = result.sort_values('date_group').reset_index(drop=True)



In [25]:
result.head()

,date_group,visits,registrations,utm_campaign,cost
0,2023-03-01,144,60.0,advanced_algorithms_series,212
1,2023-03-02,74,320.0,advanced_algorithms_series,252
2,2023-03-03,80,279.0,advanced_algorithms_series,202
3,2023-03-04,89,288.0,advanced_algorithms_series,223
4,2023-03-05,89,53.0,advanced_algorithms_series,265


In [26]:
result.to_json('ads.json')